In [18]:
# First I am importing the libraries I need

import pandas as pd
import numpy as np

In [19]:
# Now I am loading my cleaned defender dataset

df = pd.read_csv("WC2026_Cleaned_Defenders_Final.csv")

# I am checking the first few rows to make sure the file loaded properly

print(df.head())

   Rank             Player Position         Squad  Age  Born  90s  Tackles  \
0     6           Ali Abdi       DF       Tunisia   32  1993  3.0        8   
1     7    Saud Abdulhamid       DF  Saudi Arabia   26  1999  3.0        9   
2     8  Abdulla Abdullaev       DF    Uzbekistan   28  1997  2.0        2   
3    10    Husam Abu Dahab       DF        Jordan   26  2000  2.0        3   
4    12  Mohannad Abu Taha       DF        Jordan   23  2003  2.9        2   

   Interceptions (Int)        Tournament Stage  Interceptions per 90 mins  \
0                    4  Group Stage Eliminated                   1.333333   
1                    2  Group Stage Eliminated                   0.666667   
2                    3  Group Stage Eliminated                   1.500000   
3                    0  Group Stage Eliminated                   0.000000   
4                    6  Group Stage Eliminated                   2.068966   

  Eligible Defender FBref Player ID  
0               Yes        ad7

In [20]:
# Now I am checking for possible outliers in interceptions per 90 minutes
# I am using the IQR method because it helps identify unusually high or low values

Q1 = df["Interceptions per 90 mins"].quantile(0.25)
Q3 = df["Interceptions per 90 mins"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

Q1: 0.5555555555555556
Q3: 1.3798701298701297
IQR: 0.8243145743145741
Lower limit: -0.6809163059163057
Upper limit: 2.616341991341991


In [21]:
# I am checking which defenders fall outside the IQR range
# I am not deleting them yet because I first want to inspect the values

outliers = df[
    (df["Interceptions per 90 mins"] < lower_limit) |
    (df["Interceptions per 90 mins"] > upper_limit)
]

print("Number of possible outliers:", len(outliers))

print(
    outliers[
        [
            "Player",
            "Squad",
            "90s",
            "Interceptions (Int)",
            "Interceptions per 90 mins",
            "Tournament Stage"
        ]
    ]
)

Number of possible outliers: 7
                   Player         Squad  90s  Interceptions (Int)  \
21          Yazan Al-Arab        Jordan  3.0                   11   
181         Khuliso Mudau  South Africa  4.0                   11   
182     Tarik Muharemovic   Bosnia-Herz  2.9                    8   
203     Exequiel Palacios     Argentina  1.0                    5   
205  João Paulo Fernandes    Cabo Verde  1.2                    6   
213          Stefan Posch       Austria  3.7                   10   
264         Auston Trusty           USA  1.1                    4   

     Interceptions per 90 mins        Tournament Stage  
21                    3.666667  Group Stage Eliminated  
181                   2.750000                Knockout  
182                   2.758621                Knockout  
203                   5.000000                Knockout  
205                   5.000000                Knockout  
213                   2.702703                Knockout  
264              

# The IQR method found 7 possible outliers.
# I checked these players and their values are valid football statistics,
# so I am keeping them instead of removing genuine observations.

In [22]:
# First I am separating the defenders into the two tournament groups

knockout = df[
    df["Tournament Stage"] == "Knockout"
]

group_stage = df[
    df["Tournament Stage"] == "Group Stage Eliminated"
]

print("Knockout defenders:", len(knockout))
print("Group-stage eliminated defenders:", len(group_stage))

Knockout defenders: 190
Group-stage eliminated defenders: 90


In [23]:
# I am randomly selecting 40 defenders from each group
# so both groups have equal representation in my analysis

sample_knockout = knockout.sample(
    n=40,
    random_state=42
)

sample_group_stage = group_stage.sample(
    n=40,
    random_state=42
)

In [24]:
# Now I am combining the two samples into one dataset

sample_df = pd.concat(
    [sample_knockout, sample_group_stage],
    ignore_index=True
)

# I am checking the final sample size

print("Total sample size:", len(sample_df))

print(sample_df["Tournament Stage"].value_counts())

Total sample size: 80
Tournament Stage
Knockout                  40
Group Stage Eliminated    40
Name: count, dtype: int64


In [25]:
# Finally I am saving my sampled dataset
# so I can use the same sample for the rest of my analysis

sample_df.to_csv(
    "WC2026_Defenders_Final_Sample.csv",
    index=False
)

print("Sample CSV saved successfully.")

Sample CSV saved successfully.
